# 🧩 `payments_tools` — solution to Part 1

Every rule the monitor has, in one place: the fee, what counts as large, how a payment is
converted, and how a line is printed. The project notebook holds the data and the questions —
it holds no rules at all.

Notice what is **not** here: no `print`, no database, no charts. A module defines; the notebook
decides what to do with it.

In [ ]:
"""Everything the payments monitor needs: two classes and two helpers."""

import statistics

FEE_RATE = 1.5        # what the bank charges on a currency conversion, in percent
LARGE = 1000.00       # a payment at or above this is worth looking at


def format_eur(amount):
    """Return the amount as a tidy string with thousands separators."""
    return f"{amount:,.2f} EUR"


class Payment:
    """A payment to a supplier, already in euros."""

    def __init__(self, reference, supplier, amount):
        self.reference = reference
        self.supplier = supplier
        self.amount = amount

    def total(self):
        """What actually leaves the account, in EUR."""
        return round(self.amount, 2)

    def kind(self):
        """Which sort of payment this is - the database and the charts both want it."""
        return "domestic"

    def is_large(self):
        """True when this payment is worth a second look."""
        return self.total() >= LARGE

    def describe(self):
        """One line of the report."""
        line = f"{self.reference}  {self.supplier:<16}{format_eur(self.total()):>14}"
        if self.is_large():
            line = line + "  LARGE"
        return line


class ForeignPayment(Payment):
    """A payment made in another currency: converted, then charged a fee."""

    def __init__(self, reference, supplier, amount, currency, rate):
        super().__init__(reference, supplier, amount)
        self.currency = currency
        self.rate = rate

    def total(self):
        """The converted amount, plus the bank's conversion fee."""
        converted = self.amount * self.rate
        return round(converted + converted * FEE_RATE / 100, 2)

    def kind(self):
        return "foreign"

    def describe(self):
        """The same line as any payment, with the original currency on the end."""
        return super().describe() + f"   ({self.amount:,.2f} {self.currency})"


def summarise(payments):
    """Return a dict of the count, total, average and largest of some payments."""
    totals = [payment.total() for payment in payments]
    return {
        "count": len(totals),
        "total": round(sum(totals), 2),
        "average": round(statistics.mean(totals), 2),
        "largest": max(totals),
    }